# Time Series Comparison

This notebook pulls in data from the NOAA CO-OPS NWLON stations and pulls in CORA data at those locations from the NOAA Open Data Dissemination (NODD) and plots water level time series for comparison.

## Configuration

Update these values as necessary

`station_id` is a list of NOAA ID's. One source of information is https://tidesandcurrents.noaa.gov/map/index.html?region=Texas

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CONFIGURATION - MODIFY THESE VALUES AS NEEDED
# ═══════════════════════════════════════════════════════════════

# NOAA ID's of stations to query
station_id = [
              # '8665530', # Charleston, SC
              '8774770', # Port Aransas, TX
              '8771450', # Galveston Pier 21, TX
              '8775237', # Aransas, Aransas Pass, TX
              '8775296', # Enbridge, Ingleside, TX
              '8773037', # Seadrift, TX
              '8776604', # Baffin Bay, TX
              '8775792', # Packery Channel, TX
              '8773146', # Matagorda City, TX
              '8770613', # Morgans Point, TX
              '8770777', # Manchester, TX
              '8770822', # Texas Point, TX
              '8770475', # Port Arthur, TX
              '8779770', # Port Isabel, TX
              '8736897', # Coast Guard Sector Mobile, AL
            ]

#, '8775870']


# add US coast guard, baffin bay, packery channel, madagorda city, morgans point, manchester, texas point, port arthur,

# Time period for data extraction
start_year = '2018'
start_month = '09'
start_day = '01'

end_year = '2018'
end_month = '11'
end_day = '01'


## Import python libraries.

In [ ]:
import os

import requests
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dask
import intake
# import xarray as xr
import scipy.spatial as sp
# import s3fs
# import geopy.distance
from scipy.spatial import KDTree
import folium
from folium import Marker
# from folium.plugins import HeatMap, MarkerCluster
# from branca.colormap import linear, LinearColormap
from tqdm import tqdm
from dask.diagnostics import ProgressBar

### Define Utility functions
DO NOT MODIFY

In [ ]:
def area(x1, y1, x2, y2, x3, y3):
    return (abs((x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2)) / 2.0))

In [ ]:
def define_kd_tree(ds):
    e = ds.element.values.astype(int)
    emin1 = e-1
    num_elems = len(e)
    x_vals = ds.x.values
    y_vals = ds.y.values

    xe=np.mean(x_vals[emin1],axis=1)
    ye=np.mean(y_vals[emin1],axis=1)
    tree = sp.KDTree(np.c_[xe,ye])
    areas = [area(x_vals[emin1[k][0]],y_vals[emin1[k][0]],\
                  x_vals[emin1[k][1]],y_vals[emin1[k][1]],\
                  x_vals[emin1[k][2]],y_vals[emin1[k][2]])for k in range(0, num_elems)]
    return tree, areas, e, x_vals, y_vals

In [ ]:
def find_triangle(x_vals, y_vals, e, lat, lon):
    e = ds.element.values.astype(int)-1

    k = 10
    dist, ii = tree.query([lon, lat], k=k)
    ii = ii
    triangle_i = -1

    for i in range(0, k):

      a1 = area(lon, lat,\
                x_vals[e[ii[i]][0]], y_vals[e[ii[i]][0]],\
                x_vals[e[ii[i]][1]], y_vals[e[ii[i]][1]])

      a2 = area(lon, lat,\
                x_vals[e[ii[i]][1]], y_vals[e[ii[i]][1]],\
                x_vals[e[ii[i]][2]], y_vals[e[ii[i]][2]])

      a3 = area(lon, lat,\
                x_vals[e[ii[i]][0]], y_vals[e[ii[i]][0]],\
                x_vals[e[ii[i]][2]], y_vals[e[ii[i]][2]])

      t_area = a1 + a2 + a3
      if abs(t_area - areas[ii[i]]) < 0.00000001:
        triangle_i = ii[i]+1
        break
    if(triangle_i == -1):
        print("ERROR for ", lat, lon)
    return triangle_i

## Get CORA dataset files
**Access the data on the NODD and initialize the available CORA datasets.** 

*This accesses a .yml file located on the NODD that shows which CORA output files are available to import.*

See https://noaa-nos-cora-pds.s3.amazonaws.com/index.html for the latest list

In [ ]:
# @title This accesses a .yml file located on the NODD that shows which CORA output files are available to import.
catalog = intake.open_catalog("s3://noaa-nos-cora-pds/CORA_V1.1_intake.yml",storage_options={'anon':True})
list(catalog)

***CORA-V1.1-fort.63:*** Hourly water levels: ***'zeta'*** is the water elevation variable referenced to mean sea level<br>
***CORA-V1.1-swan_DIR.63:*** Hourly mean wave direction: ***'swan_DIR'*** is the mean wave direction variable<br>
***CORA-V1.1-swan_TPS.63:*** Hourly peak wave periods: ***'swan_TPS'*** is the peak wave period variable<br>
***CORA-V1.1-swan_HS.63:*** Hourly significant wave heights: ***'swan_HS'*** is the significant wave height variable<br>
***CORA-V1.1-Grid:*** Hourly water levels interpolated from model nodes to 500-meter resolution coastal grid: ***'zeta'*** is the water elevation variable <br>

> All datasets denoted as **'-timeseries'** are optimized for pulling long time series (greater than a few days)
> For up to a few days of data, use the regular dataset (not labeled **'-timeseries'** in the catalog description)


*Now, create an xarray dataset for the CORA data that you would like to use.*<br>
<br>
Using the `.to_dask()` command with the water level dataset located in the catalog will create an xarray dataset.
There is data at 1,813,443 model nodes spanning 385,704 hours (44 years, 1979-2022).
The 'zeta' hourly water level variable is given in dimensions of time and node.


In [ ]:
ds = catalog["CORA-V1.1-fort.63"].to_dask()

## Get NWLON station data

**Create a dataframe of NWLON station ids and coordinates from the CO-OPS API where you want to do a comparison.**

In [ ]:
base_url = 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/.json'
params = {
    'type': 'waterlevels',
    'units': 'metric'
}
print(f'base_url: {base_url}, parameters: {params}')
response = requests.get(base_url, params=params)
content = response.json()

stations = content['stations']
stations_df = pd.DataFrame(stations)

# Include station name along with id, lat, lng
stations_df = stations_df[['id', 'name', 'lat', 'lng', 'state']]

# limit to the list of stations specified in the configuration section
stations_df = stations_df[stations_df['id'].isin(station_id)]

stations_df

**Loop through the station list to grab water level hourly heights for each station. Create a pandas dataframe of the time series data.**

In [ ]:
base_url = 'https://api.tidesandcurrents.noaa.gov/api/prod/datagetter'

# Initialize an empty dataframe to merge all station data
df = None

for i in range(len(stations_df)):
    station_id = stations_df.id.iloc[i]
    print(f"Requesting data for station {station_id}")

    params = {
        'begin_date': f'{start_year}{start_month}{start_day}',
        'end_date': f'{end_year}{end_month}{end_day}',
        'station': station_id,
        'product': 'hourly_height',
        'datum': 'MSL',
        'time_zone': 'gmt',
        'units': 'metric',
        'format': 'json'
    }

    print(f'base_url: {base_url}, parameters: {params}')

    response = requests.get(base_url, params=params)
    content = response.json()

    # Check if we have data in the response
    if 'data' not in content or not content['data']:
        print(f"No data available for station {station_id}")
        continue

    # Extracting data for 'time' and 'height'
    time_data = [d['t'] for d in content['data']]
    tnc_data = [d['v'] for d in content['data']]

    # Convert tnc_data water level to numeric
    tnc_data = [float(x) if x and str(x).strip() != '' else np.nan for x in tnc_data]

    # Create a temporary dataframe for this station
    temp_df = pd.DataFrame({
        'time': time_data,
        f'tnc_{station_id}': tnc_data
    })

    # Convert time to datetime and set as index
    temp_df['time'] = pd.to_datetime(temp_df['time'])
    temp_df.set_index('time', inplace=True)

    # Merge with the main dataframe
    if df is None:
        df = temp_df
    else:
        df = df.merge(temp_df, left_index=True, right_index=True, how='outer')

# Display info about the final dataframe
print(f"Final dataframe shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df

## Calculate CORA water levels

### Get CORA node Coordinates

In [ ]:
tree, areas, e, x_vals, y_vals = define_kd_tree(ds)

In [ ]:
# Extract node coordinates from the mesh

node_coords = np.c_[ds.x.values, ds.y.values]

kdtree = KDTree(node_coords) # Build the kdtree that can be queried for nearest neighbor nodes to a geographic point

# Initialize
elem = np.zeros((len(stations_df), 1), dtype=int)
elem_nodes = np.zeros((len(stations_df), 3), dtype=int)
query_point = np.zeros((len(stations_df), 2))  # Assuming you want 2 columns for lat and lon
distances = np.zeros((len(stations_df), 3))    # Assuming you want 3 nearest neighbors
nearest = np.zeros((len(stations_df), 3), dtype=int)  # Assuming you want 3 nearest neighbors
dist = np.zeros((len(stations_df), 3), dtype=float)
weights = np.zeros((len(stations_df), 3), dtype=float)

# for i in range(len(stations_df)):
for i in tqdm(range(len(stations_df)), desc="Processing stations"):
    elem[i, :] = find_triangle(x_vals, y_vals, e, stations_df.lat.iloc[i], stations_df.lng.iloc[i])
    distances[i, :], nearest[i, :] = kdtree.query([stations_df.lng.iloc[i], stations_df.lat.iloc[i]], k=3)
    nearest[i, :] = nearest[i, :] + 1

    if elem[i] == -1:
        elem_nodes[i, :] = nearest[i, :]
    else:
        elem_nodes[i,:] = e[elem[i]-1]

    x_dist = stations_df.lng.iloc[i] - x_vals[elem_nodes[i,:]]
    y_dist = stations_df.lat.iloc[i] - y_vals[elem_nodes[i,:]]
    dist[i,:] = np.sqrt(x_dist * x_dist + y_dist * y_dist)

    if np.any(dist[i,:] ==0):
        weights[i,:] = np.where(dist[i,:] == 0, 1, 0)
    else:
        weights[i,:] = 1/dask.array.sqrt(x_dist * x_dist + y_dist * y_dist)

unique_nodes = np.unique(elem_nodes) # sorted unique nodes
mapped_triangle = np.searchsorted(unique_nodes, elem_nodes) # indices of the nodes


In [ ]:
unique_nodes[mapped_triangle]

In [ ]:
concat_nodes=np.concatenate(node_coords[elem_nodes-1], axis=0)
lat_nodes=concat_nodes[:,1]
lon_nodes=concat_nodes[:,0]

df_nodes=pd.DataFrame({'Lon': lon_nodes, 'Lat': lat_nodes})
df_nodes

In [ ]:
# disabled for now. It's not particularly useful

# # Create a base map centered on the first coordinate
# map_center = [stations_df['lat'].iloc[0], stations_df['lng'].iloc[0]]
# my_map = folium.Map(location=map_center, zoom_start=8)

# # Add markers for each coordinate
# for index, row in df_nodes.iterrows():
#     folium.CircleMarker(
#         location=[row['Lat'], row['Lon']],
#     ).add_to(my_map)

# for index, row in stations_df.iterrows():
#     folium.CircleMarker(
#         location=[row['lat'], row['lng']],
#         popup=row['id'], color='red'
#     ).add_to(my_map)

# tile = folium.TileLayer(
#         tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
#         attr = 'Esri',
#         name = 'Esri Satellite',
#         overlay = False,
#         control = True
#        ).add_to(my_map)

# # Save the map to an HTML file
# # my_map.save("nodesmap.html")

# my_map

In [ ]:
# Create an interactive map showing stations and their mesh triangles

# Create a base map centered on the mean coordinates of all stations
map_center = [stations_df['lat'].mean(), stations_df['lng'].mean()]
my_map = folium.Map(location=map_center, zoom_start=7)

# Add satellite imagery as base layer
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellite',
    overlay=False,
    control=True
).add_to(my_map)

# Add OpenStreetMap layer
folium.TileLayer(
    tiles='OpenStreetMap',
    name='OpenStreetMap',
    overlay=False,
    control=True
).add_to(my_map)

# # Color palette for different stations
# colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred',
#           'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white',
#           'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# Add stations and their mesh triangles
for i in range(len(stations_df)):
    station_id = stations_df.iloc[i]['id']
    station_name = stations_df.iloc[i]['name']
    station_lat = stations_df.iloc[i]['lat']
    station_lng = stations_df.iloc[i]['lng']
    station_state = stations_df.iloc[i]['state']

    # Use a different color for each station (cycle through colors if we have more stations)
    # color = colors[i % len(colors)]
    color = 'red'

    # Add station marker
    folium.CircleMarker(
        location=[station_lat, station_lng],
        radius=6,
        popup=f"<b>Station {station_id}</b><br>{station_name}<br>{station_state}<br>Lat: {station_lat:.4f}<br>Lng: {station_lng:.4f}",
        tooltip=f"Station {station_id}: {station_name}",
        color='black',
        fillColor=color,
        fillOpacity=0.8,
        weight=2
    ).add_to(my_map)

    # Get the triangle nodes for this station
    triangle_nodes = elem_nodes[i, :] - 1  # Convert to 0-based indexing

    # Get coordinates of triangle vertices
    triangle_coords = []
    for node_idx in triangle_nodes:
        triangle_coords.append([y_vals[node_idx], x_vals[node_idx]])  # [lat, lng]

    # Close the triangle by adding the first point at the end
    triangle_coords.append(triangle_coords[0])

    # Add triangle outline
    folium.PolyLine(
        locations=triangle_coords,
        color='grey',
        weight=2,
        opacity=0.8,
        popup=f"Mesh triangle for station {station_id}"
    ).add_to(my_map)

    # Add triangle vertices as small markers
    for j, node_idx in enumerate(triangle_nodes):
        folium.CircleMarker(
            location=[y_vals[node_idx], x_vals[node_idx]],
            radius=3,
            popup=f"Node {node_idx + 1} for station {station_id}",
            color='grey',
            fillColor='black',
            fillOpacity=0.6,
            weight=1
        ).add_to(my_map)

# Add layer control
folium.LayerControl().add_to(my_map)

# Add a legend
legend_html = '''
<div style="position: fixed;
            bottom: 50px; left: 50px; width: 200px; height: auto;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:14px; padding: 10px">
<p><b>Legend</b></p>
<p><i class="fa fa-circle" style="color:red"></i> Stations</p>
<p><i class="fa fa-minus" style="color:blue"></i> Mesh Triangles</p>
<p><i class="fa fa-circle" style="color:gray"></i> Mesh Nodes</p>
</div>
'''
my_map.get_root().html.add_child(folium.Element(legend_html))

print(f"Created map with {len(stations_df)} stations and their mesh triangles")
my_map

### Calculate CORA water levels

**Average the water levels at the 3 nodes of the element containing the station coordinates for the selected time period. Append these CORA values as new columns to the dataframe.**

In [ ]:
################################################################################
# Original version. Keeping for reference
################################################################################

# %%time

# start_t = f'{start_year}-{start_month}-{start_day} 00:00:00'
# end_t = f'{end_year}-{end_month}-{end_day} 23:00:00'
# dt_range=pd.date_range(start_t, end_t, freq='h',inclusive='both')

# num_ts = len(dt_range) # number of time samples
# t = np.zeros((num_ts, 3), dtype=float) # preallocate with zeros
# zeta_point = np.zeros((num_ts), dtype=float) # preallocate with zeros
# mean_zeta = np.zeros((num_ts,len(stations_df)), dtype=float)

# # for i in range(len(stations_df)):
# for i in tqdm(range(len(stations_df)), desc="Processing stations"):
#     zeta_tslice = ds["zeta"].sel(time=slice(start_t, end_t), node=unique_nodes[mapped_triangle[i]]-1).compute()
#     t = zeta_tslice.values * weights[i]
#     zeta_point = np.sum(t, axis=1) / np.sum(weights[i])
#     mean_zeta_i = np.nanmean(zeta_tslice.values, axis=1)

#     zeta_point[np.isnan(zeta_point)] = mean_zeta_i[np.isnan(zeta_point)]

#     df['cora_'+stations_df.iloc[i,0]] = zeta_point


In [ ]:
%%time

################################################################################
# Optimized version
################################################################################

start_t = f'{start_year}-{start_month}-{start_day} 00:00:00'
end_t = f'{end_year}-{end_month}-{end_day} 23:00:00'

# Get all unique nodes needed across all stations
all_needed_nodes = []
for i in range(len(stations_df)):
    all_needed_nodes.extend(unique_nodes[mapped_triangle[i]]-1)
all_needed_nodes = np.unique(all_needed_nodes)

# Extract data for all nodes at once
print("Extracting data for all stations...")
with ProgressBar():
    zeta_all = ds["zeta"].sel(time=slice(start_t, end_t), node=all_needed_nodes).compute()

# Pre-allocate results array
results = np.zeros((len(zeta_all.time), len(stations_df)))

# Process each station using the pre-loaded data
for i in range(len(stations_df)):
    # Find which indices in zeta_all correspond to this station's nodes
    station_nodes = unique_nodes[mapped_triangle[i]]-1
    node_indices = np.searchsorted(all_needed_nodes, station_nodes)

    # Extract just this station's data
    station_data = zeta_all.values[:, node_indices]

    # Apply weights
    weighted_data = station_data * weights[i]
    zeta_point = np.sum(weighted_data, axis=1) / np.sum(weights[i])

    # Handle NaN values
    mean_zeta_i = np.nanmean(station_data, axis=1)
    zeta_point[np.isnan(zeta_point)] = mean_zeta_i[np.isnan(zeta_point)]

    # Store result
    results[:, i] = zeta_point

# Add all columns to dataframe at once
for i, station_id in enumerate(stations_df.id):
    df[f'cora_{station_id}'] = results[:, i]


## Plot the data

**Plot the time series data for the NWLON observations and CORA for comparison.**

In [ ]:
import os

ylabel = "MSL, m"

# Create subdirectory for plots
plot_dir = "plots"
os.makedirs(plot_dir, exist_ok=True)

# num_plots = min(zeta_df.shape[1], len(stations_df))
for i in range(len(stations_df)):
  # Create title using both station ID and name
  station_id = stations_df.iloc[i, 0]  # id column
  station_name = stations_df.iloc[i, 1]  # name column
  station_state = stations_df.iloc[i, 4]  # state column
  title = f"Hourly Water Levels for {station_name}, {station_state}: {station_id}"

  # Create larger figure
  plt.figure(figsize=(15, 8))
  df[['tnc_'+stations_df.id.iloc[i],'cora_'+stations_df.iloc[i,0]]].plot(figsize=(15, 8))

  plt.title(title, fontsize=14, fontweight='bold')
  plt.xlabel('Date/Time (GMT)', fontsize=12)
  plt.ylabel(ylabel, fontsize=12)

  # Improve datetime label formatting
  plt.xticks(rotation=45, ha='right')

  # Add grid for better readability
  plt.grid(True, alpha=0.3)

  # Add legend with better positioning
  plt.legend(['TNC Observed', 'CORA Model'], loc='best')

  # Add date range annotation
  date_range_text = f"Data Period: {start_year}-{start_month}-{start_day} to {end_year}-{end_month}-{end_day}"
  plt.annotate(date_range_text, xy=(0.02, 0.98), xycoords='axes fraction',
               fontsize=10, ha='left', va='top',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgray', alpha=0.7))

  plt.tight_layout()

  # Create filename with time range in YYYY-MM format and save to subdirectory
  time_range = f"{start_year}-{start_month}_to_{end_year}-{end_month}"
  filename = f"{station_id}_{time_range}_water_levels.png"
  filepath = os.path.join(plot_dir, filename)
  plt.savefig(filepath, dpi=300, bbox_inches='tight')
  print(f"Saved plot: {filepath}")

plt.show()